In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
from torch import nn

/home/bigtech/anaconda3/envs/ET_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = AutoModel.from_pretrained('distilroberta-base')
tokenizer = AutoTokenizer.from_pretrained('distilroberta-base')

Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaModel: ['lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.bias', 'lm_head.decoder.weight', 'lm_head.layer_norm.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:
text = ["this is a text", "this is another text"]

encoded_input = tokenizer(text, padding=True, truncation=True, return_tensors='pt')

print(encoded_input)

print()

print(tokenizer.convert_ids_to_tokens(encoded_input['input_ids'][0]))
print(tokenizer.convert_ids_to_tokens(encoded_input['input_ids'][1]))

{'input_ids': tensor([[   0, 9226,   16,   10, 2788,    2],
        [   0, 9226,   16,  277, 2788,    2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1]])}

['<s>', 'this', 'Ġis', 'Ġa', 'Ġtext', '</s>']
['<s>', 'this', 'Ġis', 'Ġanother', 'Ġtext', '</s>']


In [41]:
outputs = model(input_ids=encoded_input['input_ids'], attention_mask=encoded_input['attention_mask'], output_hidden_states=False)

print("embeddings shape:", outputs.last_hidden_state.shape)
print()

linear = nn.Linear(outputs.last_hidden_state.shape[-1], 3)
evidence =  nn.Sequential(nn.Linear(2, 1),
                        nn.Sigmoid())

relu = nn.ReLU()

outputs_embed = outputs.last_hidden_state

h1 = linear(outputs_embed)

h2 = evidence(h1[:, :, 1:3])

h2 = relu(h2)

print("h1 shape:", h1.shape)

h2.expand_as()

embeddings shape: torch.Size([2, 6, 768])

h1 shape: torch.Size([2, 6, 3])


TypeError: expand_as() missing 1 required positional arguments: "other"

In [61]:
#print(h2)

test = h2.expand_as(outputs_embed)

#print(test.shape)

targets = torch.tensor([[1, 0, 1, 0, 1, 0],
                        [1, 0, 1, 0, 1, 0]]).unsqueeze(2)
#print(targets)
#print(targets.shape)

test_2 = (test*targets)

print(test_2.sum(dim=1).shape)

torch.Size([2, 768])
